# Visual 6: Predictive Power Analysis - Impact of Factors on Austin Property Prices

This visualization analyzes the relative importance and impact of various property features on Austin housing prices. Using a comprehensive approach combining correlation analysis and feature importance metrics, we identify which factors have the most significant predictive power for property values.

**Key Features:**
- Interactive bubble chart showing feature importance vs. correlation
- Hover tooltips with detailed statistics
- Zoom and pan capabilities for detailed exploration
- Color-coded by impact strength
- Filterable by factor type

In [ ]:
%pip install scikit-learn

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load and Prepare Data
# Load the cleaned Austin housing data
df = pd.read_csv('../data/austin_housing_cleaned.csv')

# Display basic information about the dataset
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Feature Engineering and Data Preparation
# Select relevant features for analysis
feature_columns = [
    'livingAreaSqFt', 'numOfBedrooms', 'numOfBathrooms', 'numOfStories',
    'garageSpaces', 'parkingSpaces', 'yearBuilt', 'propertyTaxRate',
    'avgSchoolRating', 'avgSchoolDistance', 'hasGarage', 'hasAssociation'
]

# Create a clean dataset with only numeric features and target
df_analysis = df[feature_columns + ['latestPrice']].copy()

# Convert boolean columns to numeric
df_analysis['hasGarage'] = df_analysis['hasGarage'].astype(int)
df_analysis['hasAssociation'] = df_analysis['hasAssociation'].astype(int)

# Remove any rows with missing values
df_analysis = df_analysis.dropna()

print(f"Analysis dataset shape: {df_analysis.shape}")
print(f"\nData types:")
print(df_analysis.dtypes)

In [ ]:
# Calculate Correlation and Feature Importance
# Prepare features and target
X = df_analysis[feature_columns]
y = df_analysis['latestPrice']

# Standardize features for better model performance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calculate Pearson correlation coefficients
correlations = {}
p_values = {}
for col in feature_columns:
    corr, p_val = pearsonr(df_analysis[col], df_analysis['latestPrice'])
    correlations[col] = corr
    p_values[col] = p_val

# Train Random Forest model for feature importance
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1)
rf_model.fit(X_scaled, y)

# Get feature importance
feature_importance = dict(zip(feature_columns, rf_model.feature_importances_))

# Create comprehensive analysis dataframe
analysis_results = pd.DataFrame({
    'Feature': feature_columns,
    'Correlation': [correlations[col] for col in feature_columns],
    'P_Value': [p_values[col] for col in feature_columns],
    'Importance': [feature_importance[col] for col in feature_columns]
})

# Calculate absolute correlation for impact strength
analysis_results['Abs_Correlation'] = analysis_results['Correlation'].abs()

# Calculate combined impact score (weighted average of importance and abs correlation)
analysis_results['Impact_Score'] = (
    0.6 * analysis_results['Importance'] + 
    0.4 * analysis_results['Abs_Correlation']
)

# Categorize features
def categorize_feature(feature):
    if feature in ['livingAreaSqFt', 'numOfBedrooms', 'numOfBathrooms', 'numOfStories']:
        return 'Property Size'
    elif feature in ['garageSpaces', 'parkingSpaces', 'hasGarage']:
        return 'Parking & Garage'
    elif feature in ['avgSchoolRating', 'avgSchoolDistance']:
        return 'School Quality'
    elif feature in ['yearBuilt']:
        return 'Property Age'
    elif feature in ['propertyTaxRate', 'hasAssociation']:
        return 'Financial & HOA'
    else:
        return 'Other'

analysis_results['Category'] = analysis_results['Feature'].apply(categorize_feature)

# Sort by impact score
analysis_results = analysis_results.sort_values('Impact_Score', ascending=False)

print("Feature Impact Analysis:")
print(analysis_results.to_string(index=False))
print(f"\nModel R² Score: {rf_model.score(X_scaled, y):.4f}")

In [ ]:
# Create Interactive Visualization
# Create human-readable feature names
feature_name_map = {
    'livingAreaSqFt': 'Living Area (sq ft)',
    'numOfBedrooms': 'Number of Bedrooms',
    'numOfBathrooms': 'Number of Bathrooms',
    'numOfStories': 'Number of Stories',
    'garageSpaces': 'Garage Spaces',
    'parkingSpaces': 'Parking Spaces',
    'yearBuilt': 'Year Built',
    'propertyTaxRate': 'Property Tax Rate',
    'avgSchoolRating': 'Avg School Rating',
    'avgSchoolDistance': 'Avg School Distance',
    'hasGarage': 'Has Garage',
    'hasAssociation': 'Has HOA'
}
analysis_results['Feature_Display'] = analysis_results['Feature'].map(feature_name_map)
analysis_results['Signed_Impact'] = analysis_results['Importance'] * np.sign(analysis_results['Correlation'])

# Create color mapping for categories
category_colors = {
    'Property Size': '#1f77b4',
    'Parking & Garage': '#ff7f0e',
    'School Quality': '#2ca02c',
    'Property Age': '#d62728',
    'Financial & HOA': '#9467bd'
}

# Axis ranges (extra breathing room for labels)
x_min_vis, x_max_vis = -0.65, 0.85
y_min_vis = 0
y_max_vis = float(analysis_results['Importance'].max() * 1.15)

fig = go.Figure()

# ---- 1) BUBBLE TRACES (by category) ----
for category in analysis_results['Category'].unique():
    category_data = analysis_results[analysis_results['Category'] == category]
    # Slightly reduce bubble scale to ease clutter
    bubble_sizes = category_data['Impact_Score'] * 850

    fig.add_trace(go.Scatter(
        x=category_data['Correlation'],
        y=category_data['Importance'],
        mode='markers',
        name=category,
        marker=dict(
            size=bubble_sizes,
            color=category_colors[category],
            opacity=0.75,
            line=dict(width=2, color='white'),
            sizemode='area'
        ),
        customdata=np.column_stack((
            category_data['Feature_Display'],
            category_data['Correlation'],
            category_data['Importance'],
            category_data['Impact_Score'],
            category_data['P_Value'],
            category_data['Category']
        )),
        hovertemplate='<b>%{customdata[0]}</b><br>' +
                      '<br><b>Predictive Metrics:</b><br>' +
                      'Correlation: %{customdata[1]:.4f}<br>' +
                      'Feature Importance: %{customdata[2]:.4f}<br>' +
                      'Combined Impact Score: %{customdata[3]:.4f}<br>' +
                      'P-Value: %{customdata[4]:.2e}<br>' +
                      'Category: %{customdata[5]}<br>' +
                      '<extra></extra>',
        showlegend=True
    ))

# ---- 2) QUADRANT-AWARE SPIRAL LABEL PLACEMENT + CONNECTOR LINES ----
x_vals = analysis_results['Correlation'].to_numpy()
y_vals = analysis_results['Importance'].to_numpy()
texts  = analysis_results['Feature_Display'].tolist()

x_span = (x_max_vis - x_min_vis)
y_span = (y_max_vis - y_min_vis)

# Convert text length to approximate data-space bbox
font_size_px = 11
char_w = 0.020 * x_span           # Moderate increase
line_h = 0.055 * y_span           # Moderate increase

def label_bbox(x, y, text):
    # cap min width so short labels still get space
    w = max(8, len(text)) * char_w * 0.55
    h = line_h
    return (x - w/2, x + w/2, y - h/2, y + h/2)

def boxes_overlap(b1, b2, padding=0.025):
    """Check if two boxes overlap with padding buffer"""
    x1_min, x1_max, y1_min, y1_max = b1
    x2_min, x2_max, y2_min, y2_max = b2
    # Add padding to prevent labels from being too close
    pad_x = padding * x_span
    pad_y = padding * y_span
    return not (x1_max + pad_x < x2_min or x2_max + pad_x < x1_min or 
                y1_max + pad_y < y2_min or y2_max + pad_y < y1_min)

# Initial outward “quadrant” vectors (push toward open space first)
def quadrant_unit(x0, y0):
    dx = 1 if x0 >= 0 else -1
    dy = 1 if y0 >= 0 else -1
    return dx, dy

# Spiral params (improved separation)
golden_angle = np.deg2rad(137.507764)
r0 = 0.030 * max(x_span, y_span)   # start farther
dr = 0.020 * max(x_span, y_span)   # larger step

# Sort densest region first so later labels avoid them
dens_order = np.argsort(
    np.abs(y_vals - np.median(y_vals)) + np.abs(x_vals - np.median(x_vals))
)

label_x = np.zeros_like(x_vals, dtype=float)
label_y = np.zeros_like(y_vals, dtype=float)
label_pos = []   # for adaptive text alignment
placed_boxes = []

margin_x = 0.05 * x_span
margin_y = 0.05 * y_span

for idx in dens_order:
    x0, y0, txt = x_vals[idx], y_vals[idx], texts[idx]

    # Start by pushing in quadrant's outward direction
    qdx, qdy = quadrant_unit(x0, y0)
    cand_x = x0 + qdx * 0.8 * r0
    cand_y = y0 + qdy * 0.8 * r0
    cand_x = max(x_min_vis + margin_x, min(cand_x, x_max_vis - margin_x))
    cand_y = max(y_min_vis + margin_y, min(cand_y, y_max_vis - margin_y))
    cand_box = label_bbox(cand_x, cand_y, txt)

    # Spiral search if still colliding
    k = 0
    max_iter = 1000
    while any(boxes_overlap(cand_box, b) for b in placed_boxes) and k < max_iter:
        r = r0 + dr * k
        theta = k * golden_angle
        # Spiral around preferred outward direction
        cand_x = x0 + qdx * r * np.cos(theta)
        cand_y = y0 + qdy * r * np.sin(theta)
        # Keep inside margins
        cand_x = max(x_min_vis + margin_x, min(cand_x, x_max_vis - margin_x))
        cand_y = max(y_min_vis + margin_y, min(cand_y, y_max_vis - margin_y))
        cand_box = label_bbox(cand_x, cand_y, txt)
        k += 1

    label_x[idx], label_y[idx] = cand_x, cand_y
    placed_boxes.append(cand_box)
    # Store relative position for text alignment later
    label_pos.append((idx, np.sign(cand_x - x0), np.sign(cand_y - y0)))

# Build connector lines with gentle elbows
line_x, line_y = [], []
for x0, y0, lx, ly in zip(x_vals, y_vals, label_x, label_y):
    midx = (x0*2 + lx) / 3.0  # small elbow closer to the point
    midy = (y0*2 + ly) / 3.0
    line_x += [x0, midx, lx, None]
    line_y += [y0, midy, ly, None]

fig.add_trace(go.Scatter(
    x=line_x, y=line_y,
    mode='lines',
    line=dict(color='rgba(0,0,0,0.35)', width=1),
    hoverinfo='skip',
    showlegend=False
))

# Decide text alignment based on where the label ended
# (labels to the right are left-aligned; to the left are right-aligned)
textpositions = []
for idx, sdx, sdy in sorted(label_pos):  # keep order aligned to arrays
    if sdx >= 0.5:
        textpositions.append('middle left')
    elif sdx <= -0.5:
        textpositions.append('middle right')
    elif sdy >= 0.5:
        textpositions.append('top center')
    else:
        textpositions.append('bottom center')

# Single text trace
fig.add_trace(go.Scatter(
    x=label_x,
    y=label_y,
    mode='text',
    text=texts,
    textposition=textpositions,
    textfont=dict(size=11, color='black'),
    hoverinfo='skip',
    showlegend=False
))

# ---- 3) LAYOUT, QUADRANTS & ANNOTATIONS ----
fig.update_layout(
    title={
        'text': '<b>Impact of Property Factors on Austin Housing Prices</b><br>' +
                '<sub>Feature Importance vs. Correlation Analysis | Bubble Size = Combined Impact Score</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    xaxis=dict(
        title=dict(
        text="<b>Correlation with Property Price</b>",
        font=dict(size=14, color="#34495e")
        ),
        showgrid=True, gridwidth=1, gridcolor="lightgray",
        zeroline=True, zerolinewidth=2, zerolinecolor="black",
        range=[x_min_vis, x_max_vis]
    ),
    yaxis=dict(
        title=dict(
        text="<b>Random Forest Feature Importance</b>",
        font=dict(size=14, color="#34495e")
        ),
        showgrid=True, gridwidth=1, gridcolor="lightgray",
        zeroline=True, zerolinewidth=2, zerolinecolor="gray",
        range=[y_min_vis, y_max_vis]
    ),
    plot_bgcolor='#f8f9fa',
    paper_bgcolor='white',
    hovermode='closest',
    legend=dict(
        title=dict(text='<b>Factor Categories</b>', font=dict(size=12)),
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.02,
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='gray',
        borderwidth=1
    ),
    width=1200,
    height=750,
    annotations=[
        dict(x=0.50, y=0.36, xref='x', yref='y',
             text='<i><br></i>',
             showarrow=False, font=dict(size=12, color='green'),
             bgcolor='rgba(144, 238, 144, 0.3)', borderpad=5),
        dict(x=-0.40, y=0.36, xref='x', yref='y',
             text='<i><br></i>',
             showarrow=False, font=dict(size=12, color='orange'),
             bgcolor='rgba(255, 200, 124, 0.3)', borderpad=5),
        dict(x=0.50, y=-0.04, xref='x', yref='y',
             text='<i><br></i>',
             showarrow=False, font=dict(size=12, color='blue'),
             bgcolor='rgba(173, 216, 230, 0.3)', borderpad=5),
        dict(x=-0.40, y=-0.04, xref='x', yref='y',
             text='<i><br></i>',
             showarrow=False, font=dict(size=12, color='gray'),
             bgcolor='rgba(211, 211, 211, 0.3)', borderpad=5),
        dict(
            x=0.02, y=0.98, xref='paper', yref='paper',
            text='<b>Key Insights:</b><br>'
                 '• Bubble size represents combined predictive power<br>'
                 '• Positive correlation = higher values increase price<br>'
                 '• Feature importance shows model prediction weight<br>'
                 '• Click legend items to filter by category<br>'
                 '• Hover for detailed statistics',
            showarrow=False, font=dict(size=10, color='#2c3e50'),
            align='left', bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='#3498db', borderwidth=2, borderpad=8
        )
    ]
)

# Interactivity config
config = {
    'displayModeBar': True,
    'displaylogo': False,
    'modeBarButtonsToAdd': ['select2d', 'lasso2d'],
    'modeBarButtonsToRemove': ['autoScale2d'],
    'toImageButtonOptions': {
        'format': 'png',
        'filename': 'austin_housing_predictive_factors',
        'height': 700,
        'width': 1200,
        'scale': 2
    }
}

fig.show(config=config)

## Export Visual as JSON

In [ ]:
import plotly.io as pio
import os

# Make sure your export folder exists
os.makedirs("../reports/exports", exist_ok=True)

# Save the figure as a full Plotly JSON (includes layout, data, and config)
pio.write_json(fig, "../reports/exports/Housing_Impact_Visual.plotly.json", pretty=True)

print("Saved to exports/Housing_Impact_Visual.plotly.json")
